In [0]:
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    ArrayType,
    TimestampType
)

# Storage configuration — aligned with Universal Bronze and Silver
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

# Existing schema configuration
SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"

# New Silver dataset configuration
SILVER_DATASET_CONFIG_PATH = (
    BASE_PATH + "metadata/silver_dataset_config/"
)
dbutils.widgets.text("dataset_name","google_ads_campaigns")
DATASET_NAME = dbutils.widgets.get("dataset_name")

print("Schema config:", SCHEMA_CONFIG_PATH)
print("Silver dataset config:", SILVER_DATASET_CONFIG_PATH)
print("Dataset name:", DATASET_NAME)

Schema config: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/schema_config/
Silver dataset config: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/silver_dataset_config/
Dataset name: google_ads_campaigns


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS nestle_sales_databricks.metadata
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS nestle_sales_databricks.metadata.silver_dataset_config
USING DELTA
LOCATION '{SILVER_DATASET_CONFIG_PATH}'
""")

DataFrame[]

In [0]:
x=spark.table('nestle_sales_databricks.metadata.silver_dataset_config')
display(x)                                             


dataset_name,source_system,is_active,config_status,load_strategy,scd_type,business_key_columns,sequence_column,deduplication_enabled,partition_columns,scd2_tracked_columns,effective_from_column,effective_to_column,is_current_column,created_at,updated_at
google_ads_campaigns,google_ads,true,APPROVED,INCREMENTAL,2,"List(customer_id, campaign_id)",ingestion_timestamp,true,List(),"List(campaign_name, campaign_status, advertising_channel_type, advertising_channel_sub_type, campaign_budget_resource_name, bidding_strategy_type, start_date, end_date, campaign_primary_status, serving_status, target_country, business_brand, product_category, business_objective)",effective_from,effective_to,is_current,2026-11-14T14:35:38.331Z,2026-11-14T14:35:38.331Z


In [0]:
%sql
select * from nestle_sales_databricks.metadata.schema_config

dataset_name,source_system,schema_ddl,schema_version,is_active,created_at,updated_at
google_ads_campaigns,google_ads,"customer_id STRING, campaign_id STRING, campaign_resource_name STRING, campaign_name STRING, campaign_status STRING, advertising_channel_type STRING, advertising_channel_sub_type STRING, campaign_budget_resource_name STRING, bidding_strategy_type STRING, start_date STRING, end_date STRING, campaign_primary_status STRING, serving_status STRING, target_country STRING, business_brand STRING, product_category STRING, business_objective STRING, source_extract_timestamp STRING",1,true,2026-10-29T04:14:08.723Z,2026-10-29T04:14:08.723Z
products,product_api,"product_id BIGINT, product_name STRING, category STRING, brand STRING, price DOUBLE",1,true,2026-10-29T03:46:14.861Z,2026-10-29T03:46:14.861Z


In [0]:
if not DeltaTable.isDeltaTable(spark, SCHEMA_CONFIG_PATH):
    raise RuntimeError(
        f"Schema config table not found: {SCHEMA_CONFIG_PATH}"
    )

schema_config_df = (
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
)

active_schema = (
    schema_config_df
    .filter(
        (F.lower(F.trim(F.col("dataset_name"))) == DATASET_NAME)
        & (F.col("is_active") == True)
    )
    .select("dataset_name", "source_system")
    .collect()
)

if len(active_schema) != 1:
    raise ValueError(
        f"Expected exactly one active schema for {DATASET_NAME}, "
        f"found {len(active_schema)}"
    )

SOURCE_SYSTEM = active_schema[0]["source_system"]

print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)

Dataset: google_ads_campaigns
Source system: google_ads


In [0]:
silver_dataset_config_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("source_system", StringType(), True),

    StructField("is_active", BooleanType(), False),
    StructField("config_status", StringType(), False),

    StructField("load_strategy", StringType(), False),
    StructField("scd_type", StringType(), False),

    StructField(
        "business_key_columns",
        ArrayType(StringType()),
        True
    ),

    StructField("sequence_column", StringType(), True),
    StructField("deduplication_enabled", BooleanType(), False),

    StructField(
        "partition_columns",
        ArrayType(StringType()),
        True
    ),

    StructField(
        "scd2_tracked_columns",
        ArrayType(StringType()),
        True
    ),

    StructField("effective_from_column", StringType(), True),
    StructField("effective_to_column", StringType(), True),
    StructField("is_current_column", StringType(), True),

    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

print("Silver dataset configuration schema defined.")

Silver dataset configuration schema defined.


In [0]:
if not DeltaTable.isDeltaTable(
    spark,
    SILVER_DATASET_CONFIG_PATH
):

    empty_df = spark.createDataFrame(
        [],
        silver_dataset_config_schema
    )

    (
        empty_df.write
        .format("delta")
        .mode("ignore")
        .save(SILVER_DATASET_CONFIG_PATH)
    )

    print("Silver dataset configuration table created.")

else:
    print("Silver dataset configuration table already exists.")

Silver dataset configuration table already exists.


In [0]:

#now = datetime.now(timezone.utc)

dataset_configs = [
    {
        "dataset_name": "google_ads_campaigns",

        # Dataset approval
        "is_active": True,
        "config_status": "APPROVED",

        # Loading strategy
        "load_strategy": "INCREMENTAL",
        "scd_type": "2",

        # Business keys
        "business_key_columns": [
            "customer_id",
            "campaign_id"
        ],

        # Observed-time SCD2 sequence
        "sequence_column": "ingestion_timestamp",

        "deduplication_enabled": True,

        "partition_columns": [],

        # Columns whose changes create new SCD2 versions
        "scd2_tracked_columns": [
            "campaign_name",
            "campaign_status",
            "advertising_channel_type",
            "advertising_channel_sub_type",
            "campaign_budget_resource_name",
            "bidding_strategy_type",
            "start_date",
            "end_date",
            "campaign_primary_status",
            "serving_status",
            "target_country",
            "business_brand",
            "product_category",
            "business_objective"
        ],

        # SCD2 effective-date columns
        "effective_from_column": "2026-01-01",
        "effective_to_column": "effective_to",
        "is_current_column": True
    }
]


In [0]:
# ============================================================
# 7. VALIDATE DATASET CONFIGURATION
# ============================================================

valid_load_strategies = [
    "INCREMENTAL",
    "FULL_LOAD",
    "BACKFILL"
]

valid_scd_types = ["0", "1", "2"]

for config in dataset_configs:

    if config["load_strategy"] not in valid_load_strategies:
        raise ValueError(
            f"Invalid load strategy for {config['dataset_name']}"
        )

    if config["scd_type"] not in valid_scd_types:
        raise ValueError(
            f"Invalid SCD type for {config['dataset_name']}"
        )

    if not config["business_key_columns"]:
        print(
            f"WARNING: No business keys configured for "
            f"{config['dataset_name']}"
        )

In [0]:
# ============================================================
# 8. BUILD DATAFRAME FOR INSERTION
# ============================================================

now = spark.sql("SELECT current_timestamp()").first()[0]

rows = []

for config in dataset_configs:

    rows.append((
        config["dataset_name"],
        SOURCE_SYSTEM if config["dataset_name"] == DATASET_NAME
        else None,

        config["is_active"],
        config["config_status"],

        config["load_strategy"],
        config["scd_type"],

        config["business_key_columns"],
        config["sequence_column"],

        config["deduplication_enabled"],

        config["partition_columns"],
        config["scd2_tracked_columns"],

        config["effective_from_column"],
        config["effective_to_column"],
        config["is_current_column"],

        now,
        now
    ))

source_df = spark.createDataFrame(
    rows,
    silver_dataset_config_schema
)

In [0]:
target = DeltaTable.forPath(
    spark,
    SILVER_DATASET_CONFIG_PATH
)

(
    target.alias("target")
    .merge(
        source_df.alias("source"),
        "target.dataset_name = source.dataset_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver dataset configuration upsert completed.")

Silver dataset configuration upsert completed.


In [0]:
result_df = (
    spark.read
    .format("delta")
    .load(SILVER_DATASET_CONFIG_PATH)
)

display(
    result_df.filter(
        F.col("dataset_name") == DATASET_NAME
    )
)

dataset_name,source_system,is_active,config_status,load_strategy,scd_type,business_key_columns,sequence_column,deduplication_enabled,partition_columns,scd2_tracked_columns,effective_from_column,effective_to_column,is_current_column,created_at,updated_at
google_ads_campaigns,google_ads,true,APPROVED,INCREMENTAL,2,"List(customer_id, campaign_id)",ingestion_timestamp,true,List(),"List(campaign_name, campaign_status, advertising_channel_type, advertising_channel_sub_type, campaign_budget_resource_name, bidding_strategy_type, start_date, end_date, campaign_primary_status, serving_status, target_country, business_brand, product_category, business_objective)",2026-01-01,effective_to,true,2026-11-16T16:11:32.530Z,2026-11-16T16:11:32.530Z
